# 05 — Verify the v3 versus XGBoost resume statistic

This notebook reproduces the statement: **v3 detected 46% of held-out high-PM2.5 events, compared with 7% for XGBoost.**

Both models train only on targets before August 1, 2026. August remains an untouched test period, so the comparison does not leak test outcomes into training. A high event means the actual and predicted PM2.5 are both above 35.4 µg/m³.

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / 'common').exists() else ROOT.parent
sys.path.insert(0, str(ROOT / 'modeling'))

HIGH_PM25 = 35.4
TRAIN_EPOCHS = 1  # selected using validation data in notebook 04
WINDOWS_PATH = ROOT / 'modeling' / 'artifacts' / 'v3' / 'windows.npz'
XGBOOST_PREDICTIONS = (
    ROOT / 'modeling' / 'artifacts' / 'v3' / 'xgboost_resume_predictions.npy'
)


xgboost_script = r'''
import sys
import numpy as np
from xgboost import XGBRegressor

windows = np.load(sys.argv[1])
X = windows['X']
y = windows['y']
development = np.concatenate([windows['train'], windows['validation']])
test = windows['test']
model = XGBRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=0,
)
model.fit(X[development].reshape(len(development), -1), y[development])
prediction = model.predict(X[test].reshape(len(test), -1))
np.save(sys.argv[2], prediction)
'''
subprocess.run(
    [sys.executable, '-c', xgboost_script,
     str(WINDOWS_PATH), str(XGBOOST_PREDICTIONS)],
    check=True,
)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from dataset import SequenceDataset, smoke_sampler
from model import LSTMRegressor
from splits import apply_scaler, fit_feature_scaler
from windowing import FEATURE_COLS

if not WINDOWS_PATH.exists():
    raise FileNotFoundError(
        'Run notebooks 01_eda.ipynb and 02_windows.ipynb first.'
    )

windows = np.load(WINDOWS_PATH)
X = windows['X']
y = windows['y']
current_pm25 = windows['current']
development = np.concatenate([windows['train'], windows['validation']])
test = windows['test']

print(f'development windows: {len(development):,}')
print(f'held-out August windows: {len(test):,}')
print(f'actual high-PM2.5 events: {int((y[test] > HIGH_PM25).sum())}')

In [ ]:
# Reproduce the deterministic evaluation model from notebook 04.
torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cpu')

scaler = fit_feature_scaler(X[development])
X_development = apply_scaler(scaler, X[development])
X_test = apply_scaler(scaler, X[test])

train_dataset = SequenceDataset(
    X_development, y[development], current_pm25[development]
)
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    sampler=smoke_sampler(y[development], current_pm25[development]),
)

v3 = LSTMRegressor(
    n_features=len(FEATURE_COLS),
    hidden=64,
    layers=1,
    dropout=0.0,
    residual=True,
).to(device)
optimizer = torch.optim.Adam(v3.parameters(), lr=1e-3)

def smoke_loss(prediction, target):
    weights = torch.ones_like(target)
    weights = torch.where(target > 12.0, 1.5, weights)
    weights = torch.where(target > 35.4, 3.0, weights)
    weights = torch.where(target > 55.4, 5.0, weights)
    underprediction = (target > 35.4) & (prediction < target)
    weights = torch.where(underprediction, weights * 1.25, weights)
    raw = F.huber_loss(
        prediction, target, reduction='none', delta=10.0
    )
    return (raw * weights).sum() / weights.sum()

for epoch in range(TRAIN_EPOCHS):
    v3.train()
    losses = []
    for X_batch, y_batch, current_batch in train_loader:
        optimizer.zero_grad()
        prediction = v3(X_batch, current_batch)
        loss = smoke_loss(prediction, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(v3.parameters(), max_norm=5.0)
        optimizer.step()
        losses.append(loss.item())
    print(f'epoch {epoch + 1}: loss={np.mean(losses):.3f}')

v3.eval()
with torch.no_grad():
    v3_prediction = v3(
        torch.from_numpy(X_test).float(),
        torch.from_numpy(current_pm25[test]).float().unsqueeze(1),
    ).numpy().ravel()

In [ ]:
# Load the XGBoost predictions produced from the same development split.
xgboost_prediction = np.load(XGBOOST_PREDICTIONS)
test_target = y[test]
actual_high = test_target > HIGH_PM25
high_count = int(actual_high.sum())

def event_metrics(name, prediction):
    detected = int((prediction[actual_high] > HIGH_PM25).sum())
    recall = detected / high_count
    high_mae = float(
        np.abs(prediction[actual_high] - test_target[actual_high]).mean()
    )
    return {
        'model': name,
        'actual_high_events': high_count,
        'events_detected': detected,
        'event_recall': recall,
        'high_event_MAE': high_mae,
    }

results = pd.DataFrame([
    event_metrics('v3 residual LSTM', v3_prediction),
    event_metrics('XGBoost', xgboost_prediction),
])

display(results.style.format({
    'event_recall': '{:.1%}',
    'high_event_MAE': '{:.2f}',
}))

v3_result = results.iloc[0]
xgb_result = results.iloc[1]
print(
    f"V3 detected {v3_result.events_detected:.0f}/{high_count} "
    f"high events ({v3_result.event_recall:.1%}); "
    f"XGBoost detected {xgb_result.events_detected:.0f}/{high_count} "
    f"({xgb_result.event_recall:.1%})."
)
print(
    f"Resume wording: Detected {v3_result.event_recall:.0%} of "
    f"high-PM2.5 events in held-out test data, compared with "
    f"{xgb_result.event_recall:.0%} for XGBoost."
)